# Initialise Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import lit, trim, col

# Read from Bronze table

In [0]:
df = spark.read.table("workspace.bronze.crm_prd_info")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

## Product Key Parsing
###### This code extracts a category ID from the first 5 characters of prd_key (replacing - with _), then updates prd_key to keep only the remaining portion of the string. 

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

## Cost Cleanup
###### This code replaces NULL values in prd_cost with 0 while keeping existing values unchanged.

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

## Product Line Normalisation

In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountine")
        .when(F.upper(col("prd_line")) == "R", "Road")
        .when(F.upper(col("prd_line")) == "S", "Other Sales")
        .when(F.upper(col("prd_line")) == "T", "Touring")
        .otherwise("n/a")
    )
)

## Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

## Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id" : "product_id",
    "cat_id" : "category_id",
    "prd_key" : "product_key",
    "prd_nm" : "product_name",
    "prd_cost" : "product_cost",
    "prd_line" : "product_line",
    "prd_start_dt" : "start_date",
    "prd_end_dt" : "end_date" 
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Writing to Silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

# Confirmation Check

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10